# Forge: build a block against simulated Mojito data

This notebook is a workbench, not a tutorial: the reasoning behind the
`Block` contract is in `examples/create_block.ipynb`. Copy the `forge/`
folder once per block, then:

1. Fill in the `# EDIT` methods of `ForgeBlock` in `block.py`.
2. Run this notebook top to bottom: it loads a window of the simulated L1
   file named by `ENCHILADA_MOJITO_L1`, exercises the block by hand and runs
   `check_block`.
3. Run `wheel.ipynb` for a short campaign.

Edits to `block.py` take effect cell by cell (autoreload patches the class in
place; after changing a dataclass field, rerun the cell that builds the
instance). The cells to change here are the settings and the lines marked
`# EDIT`; the asserts are the contract and stay.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from copy import deepcopy

import numpy as np

from data import analytic_covariance, load_window, white_psd
from enchilada import Block, FixedNoiseBlock, transform
from enchilada.testing import check_block

MOJITO_PATH = os.environ.get("ENCHILADA_MOJITO_L1")
if not MOJITO_PATH:
    raise SystemExit(
        "set ENCHILADA_MOJITO_L1 to a Mojito-layout L1 file "
        "(in Jupyter: %env ENCHILADA_MOJITO_L1=/path/to/file.h5, then rerun this cell)"
    )
DURATION_S = float(os.environ.get("FORGE_DURATION_S", 86400.0))
FACTOR = int(os.environ.get("FORGE_FACTOR", 10))
DOMAIN = "frequency"  # the grid ForgeBlock renders on; see _check_residual
# DOMAIN = "wdm" needs the row count: it and the decimated window length divided
# by it must both be even integers (64 rows for the default day at factor 10
# from a 4 Hz file).
NUM_FREQUENCY_DIVISIONS = None
CHANNELS = ("A", "E")

window = load_window(
    MOJITO_PATH, factor=FACTOR, duration_s=DURATION_S, channels=CHANNELS
)
# A white stand-in for the simulation's noise model, at the window's own level
# (for white native noise about 12 % below the in-band PSD: decimation rolls
# off the top fifth of the band; a loud signal in the window inflates it).
# The GB example's `FixedLISANoise().psd` (copy `examples/gb_model.py` beside
# this notebook; needs lisaanalysistools) is a TDI 1.5 sensitivity; Mojito's
# channels are TDI 2.0, so convert it before using it as `PSD`.
LEVEL = 2 * np.mean([window.channel_data[name].var() for name in CHANNELS]) / window.fs
PSD = white_psd(LEVEL)
residual = transform(window, DOMAIN, num_frequency_divisions=NUM_FREQUENCY_DIVISIONS)
covariance = analytic_covariance(residual, PSD)
print(
    f"{residual.data_domain} grid, channels {residual.channel_names}, "
    f"N = {residual.N}, fs = {residual.fs} Hz, Tobs = {residual.Tobs} s"
)

## 1. The model: fill in `block.py`

`ForgeBlock` is a dataclass whose fields are the model's fixed configuration;
its nested `State` holds diagnostics that never reach the ledger. The methods
to replace carry an `# EDIT` marker:

- `state_from_parameters(parameters, reference_data)` renders the signal on
  the block's grid from one parameter set; `check_block` and `L2Data` call it. The placeholder is a sinusoid at `frequency_hz` as
  `dt * rfft` (`_basis`), the same in every channel.
- `initial_sample(conditional_residual, covariance, *, rng)` draws the start
  from the prior, or renders `start_amplitude`.
- `sample(conditional_residual, covariance, current_block_state, *, rng)`
  draws the exact conjugate posterior; replace it with your sampler and keep
  returning a complete state after acceptance and rejection alike.
- `_check_residual` and `_check_covariance` say what the model needs from its
  inputs; keep them honest so misregistration fails loudly.
- `rfft_weights` and `inner_product` at module level carry the Whittle
  normalisation on the frequency grid; a replacement likelihood reuses them.

Keep `State.num_sample_calls`: the cells below assert on it.

In [ ]:
from block import ForgeBlock

block = ForgeBlock(name="forge", frequency_hz=1e-3)  # EDIT: your block's configuration
assert isinstance(block, Block)
print(block)

## 2. Exercise the block by hand

`initial_sample` then `sample`, called the way a Wheel calls them: by keyword,
with the residual on the block's grid, the covariance on that grid and an RNG.
`check_state` is the contract in miniature for a signal block; the printed
values are the model.

In [ ]:
def check_state(state):
    assert set(state.template) == set(residual.channel_names)
    for name in residual.channel_names:
        assert state.template[name].shape == residual.channel_data[name].shape
        assert np.isfinite(state.template[name]).all()
    assert isinstance(state.parameters, dict) and state.parameters


rng = np.random.default_rng(0)
calls_before = block.state.num_sample_calls
start = block.initial_sample(
    conditional_residual=residual, covariance=covariance, rng=rng
)
check_state(start)
moved = block.sample(
    conditional_residual=residual,
    covariance=covariance,
    current_block_state=start,
    rng=rng,
)
check_state(moved)
assert block.state.num_sample_calls == calls_before + 1
print("start     ", start.parameters)
print("one sample", moved.parameters)
print(block.state)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None and residual.data_domain == "frequency":
    frequencies = np.fft.rfftfreq(residual.N, residual.dt)[1:]
    channel = CHANNELS[0]
    figure, axes = plt.subplots(figsize=(8, 4))
    axes.loglog(
        frequencies,
        np.abs(residual.channel_data[channel][1:]),
        label=f"residual {channel}",
    )
    axes.loglog(
        frequencies, np.abs(moved.template[channel][1:]), label=f"template {channel}"
    )
    axes.loglog(
        frequencies,
        np.sqrt(covariance.pair(channel, channel).real[1:]),
        label="noise model rms",
    )
    axes.set_xlabel("frequency [Hz]")
    axes.set_ylabel("|dt * rfft|")
    axes.legend()
    plt.show()

## 3. The contract check

`check_block` builds a scratch Wheel, registers the PSD's covariance there as
a noise block (`noise_blocks`, here one `FixedNoiseBlock`), then registers the
block it is given as a signal block, calling its `initial_sample`, and runs
`sample` for a few cycles, validating every returned state as a campaign
would. It advances that block's state, so this cell hands it a copy and this
block's counters stay put.

In [ ]:
calls_before_check = block.state.num_sample_calls
check_block(
    deepcopy(block),
    residual,
    num_cycles=3,
    role="signal",
    noise_blocks=(FixedNoiseBlock("psd", covariance),),
    random_seed=1,
)
assert block.state.num_sample_calls == calls_before_check
print("check_block passed")

## Before moving on

- Does the prior have the support your model needs, and does `initial_sample`
  draw from it (or render a configured start)?
- Is the likelihood normalised for the grid? On the frequency grid an rfft bin
  counts once at DC and Nyquist and twice elsewhere (`rfft_weights`, folded
  into `inner_product`).
- Do `_check_residual` and `_check_covariance` refuse what the model cannot
  use (another grid, a missing covariance, correlated channels)?
- Does `sample` return a complete state after a rejection too?
- Collect chains outside the block (see `wheel.ipynb`) and use a fresh
  instance per campaign.

Continue in `wheel.ipynb`.